# Views and changing data

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-12-views-and-changing-data.ipynb)

Queries read data. Production pipelines also create objects and change data: they define views, create tables,
add rows, replace partitions, and merge new records into existing tables. This chapter covers the statements
for each, and which ones need a table format that supports transactions.

Spark runs in this notebook with its built-in tables. Statements that need Delta Lake or Apache Iceberg are shown
as reference code, because those formats are not part of this notebook's setup.

## Setup

Install PySpark and start a Spark session. Tables go to a temporary warehouse directory, so no files are left in the notebook folder.

### Where you'll use it

- **Stable interfaces:** a view named `active_customers` lets analysts query one definition, and the definition changes in one place.
- **Access control:** a view that hides personal columns from a reporting role.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .config("spark.sql.warehouse.dir", "/tmp/sql-learning-warehouse")
    .getOrCreate()
)

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

## 1. Views

A **view** is a saved query with a name. A permanent view reads tables, not temporary views, so the orders are
stored in a table first, `orders_table`. Querying the view runs the saved query again, so a view stores no data.
A **temporary view** lasts for the current session. A **permanent view** is stored in the catalog, so other
sessions can use it.

In [ ]:
spark.sql("DROP VIEW IF EXISTS completed_orders")
spark.sql("DROP TABLE IF EXISTS orders_table")
spark.sql("CREATE TABLE orders_table USING PARQUET AS SELECT * FROM orders")
spark.sql("""
CREATE VIEW completed_orders AS
SELECT order_id, customer_id, amount
FROM orders_table
WHERE status = 'completed'
""")

spark.sql("SELECT COUNT(*) AS completed, SUM(amount) AS total FROM completed_orders").show()

Views keep the logic in one place. If the rule changes, you change the view, and every query that reads it picks up
the change. Views do not make a query faster by themselves, because the engine still runs the saved query.

## 2. Creating tables from a query

`CREATE TABLE ... AS SELECT` creates a table and fills it in one statement. The table is a snapshot: later changes
to `orders` do not reach it.

### Where you'll use it

- **Materialised outputs:** `CREATE TABLE AS SELECT` is how a pipeline writes a snapshot, a backfill or a reporting table.

In [ ]:
spark.sql("DROP TABLE IF EXISTS completed_snapshot")
spark.sql("""
CREATE TABLE completed_snapshot USING PARQUET AS
SELECT * FROM completed_orders
""")

spark.sql("SELECT COUNT(*) AS rows_in_snapshot FROM completed_snapshot").show()

## 3. Adding rows: INSERT INTO

`INSERT INTO ... VALUES` adds literal rows. `INSERT INTO ... SELECT` adds the rows of a query, which is how most
pipelines append a new batch.

### Where you'll use it

- **Appending daily loads:** a scheduled job adds yesterday's rows to a fact table.

In [ ]:
spark.sql("INSERT INTO completed_snapshot VALUES (15, 1, 500)")
spark.sql("""
INSERT INTO completed_snapshot
SELECT order_id, customer_id, amount FROM completed_orders WHERE order_id = 1
""")

spark.sql("SELECT COUNT(*) AS rows_after_inserts FROM completed_snapshot").show()

Appending is not idempotent: running the same `INSERT INTO ... SELECT` twice adds the rows twice. Pipelines that
can be re-run need one of the patterns below.

## 4. Replacing data: INSERT OVERWRITE

`INSERT OVERWRITE` replaces the existing rows with the result of the query. Running it again with the same input
gives the same table, which makes it the simplest idempotent load. On a partitioned table, Spark overwrites only
the partitions the query writes, which is how a daily job rewrites one day at a time.

### Where you'll use it

- **Idempotent daily partitions:** overwriting one partition makes a rerun produce the same result, which is why Spark and Hive pipelines use it.

In [ ]:
spark.sql("""
INSERT OVERWRITE TABLE completed_snapshot
SELECT order_id, customer_id, amount FROM orders WHERE amount >= 300
""")

spark.sql("SELECT COUNT(*) AS rows_after_overwrite FROM completed_snapshot").show()

## 5. UPDATE, DELETE and MERGE

Changing or removing individual rows needs a table format that records each change in a transaction log, such as
**Delta Lake**, **Apache Iceberg** or **Apache Hudi**, or a cloud warehouse table. Plain Parquet tables cannot be
updated in place, and temporary views cannot be changed at all. The statements below are reference code for those
formats:

```sql
-- change rows that match a condition
UPDATE accounts SET balance = balance + 10 WHERE id = 1;

-- remove rows that match a condition
DELETE FROM accounts WHERE balance < 60;

-- upsert: update matches, insert new keys, and remove keys missing from the source
MERGE INTO accounts AS target
USING account_updates AS source
  ON target.id = source.id
WHEN MATCHED THEN UPDATE SET target.balance = source.balance
WHEN NOT MATCHED THEN INSERT (id, balance) VALUES (source.id, source.balance)
WHEN NOT MATCHED BY SOURCE THEN DELETE;
```

Before a `MERGE`, make the source one row per key. If two source rows match the same target row, most engines
refuse the statement. The deduplication chapter shows how to pick one row per key.

## 6. Transactions

A transaction groups statements so that either all of them take effect or none do. Table formats such as Delta and
Iceberg commit each write atomically, so a reader sees the old table or the new one, never a partial write.
Spark's built-in tables and temporary views give no such guarantee. The interview concepts chapter explains
ACID and isolation levels in more detail.

## Interview notes

- **Idempotent loads.** Use `INSERT OVERWRITE` for a partition or a whole table, or `MERGE` on a business key.
  Either one gives the same result when the job runs twice.
- **Hard delete vs soft delete.** A soft delete sets a flag or an end date and keeps the row, so history survives.
  A hard delete removes the row. Analytics tables usually keep history.
- **Views vs tables.** A view is recomputed on every read; a table is computed once and stored. Use a table when
  the query is expensive or many queries read the same result.

## Practice questions

1. Create a table `completed_check` from the `completed_orders` view. How many rows does it have?
2. Append one row to `completed_check` with `order_id` 16, `customer_id` 2 and amount 400. How many rows now?
3. Using `INSERT OVERWRITE`, replace `completed_check` with the orders whose amount is 450 or more. How many rows?

## Answers

### Question 1

### Where you'll use it

- **Applying changes:** a `MERGE` applies a CDC batch of inserts, updates and deletes to a warehouse or lakehouse table.
- **Privacy requests:** deleting one customer's rows to meet a GDPR erasure request.

### Where you'll use it

- **All-or-nothing loads:** a multi-table update in an OLTP database, or a warehouse transaction that must not show readers half of a change.

In [ ]:
spark.sql("DROP TABLE IF EXISTS completed_check")
spark.sql("CREATE TABLE completed_check USING PARQUET AS SELECT * FROM completed_orders")
q = "SELECT COUNT(*) AS n FROM completed_check"
spark.sql(q).show()
assert spark.sql(q).collect() == [(11,)]

### Question 2

In [ ]:
spark.sql("INSERT INTO completed_check VALUES (16, 2, 400)")
q = "SELECT COUNT(*) AS n FROM completed_check"
spark.sql(q).show()
assert spark.sql(q).collect() == [(12,)]

### Question 3

In [ ]:
spark.sql("INSERT OVERWRITE TABLE completed_check SELECT order_id, customer_id, amount FROM orders WHERE amount >= 450")
q = "SELECT COUNT(*) AS n FROM completed_check"
spark.sql(q).show()
assert spark.sql(q).collect() == [(2,)]

In [ ]:
spark.sql("DROP TABLE IF EXISTS completed_check")
spark.sql("DROP TABLE IF EXISTS completed_snapshot")
spark.sql("DROP VIEW IF EXISTS completed_orders")
spark.sql("DROP TABLE IF EXISTS orders_table")